In [ ]:
import kagglehub
import pandas as pd

train_path = kagglehub.competition_download(
    "playground-series-s4e10",
    path="train.csv"
)
test_path = kagglehub.competition_download(
    "playground-series-s4e10",
    path="test.csv"
)
df_full_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)


ReadTimeout: HTTPSConnectionPool(host='storage.googleapis.com', port=443): Read timed out. (read timeout=5)

In [37]:
import numpy as np
df_train.head()

,id,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
29242,29242,22,72000,MORTGAGE,2.0,EDUCATION,A,10000,7.88,0.14,N,3
47036,47036,22,54000,RENT,5.0,MEDICAL,B,1500,10.99,0.03,N,3
54616,54616,24,95000,MORTGAGE,5.0,PERSONAL,A,5000,7.51,0.05,N,3
13552,13552,25,80000,MORTGAGE,1.0,VENTURE,B,5000,11.26,0.06,N,4
38408,38408,31,46300,MORTGAGE,16.0,DEBTCONSOLIDATION,A,3600,7.88,0.08,N,6


In [67]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(df_full_train, random_state=42, test_size=0.25)
df_train.reset_index(drop = True)
df_val.reset_index(drop = True)
df_full_train.reset_index(drop = True)

y_train = df_train.loan_status.values
y_val = df_val.loan_status.values
y_full = df_full_train.loan_status.values
del df_train['loan_status']
del df_val['loan_status']
del df_full_train['loan_status']


In [73]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def train(df, y_train, C = 1):
    dv = DictVectorizer(sparse=False)
    X_dict = df.drop(columns = ['id']).to_dict(orient = 'records')
    scaler = StandardScaler()
    X_train = dv.fit_transform(X_dict)

    numerical = [
        'person_age',
        'person_income',
        'person_emp_length',
        'loan_amnt',
        'loan_int_rate',
        'loan_percent_income',
        'cb_person_cred_hist_length'
    ]

    indices = [list(dv.get_feature_names_out()).index(col) for col in numerical]
    X_train[:, indices] = scaler.fit_transform(X_train[:, indices])
    model = LogisticRegression(solver='liblinear', C=с, max_iter=1000)
    model.fit(X_train, y_train)
    return dv, indices, scaler, model
def prediction(df_val, dv, indices, scaler, model):
    X_dict = df_val.drop(columns = ['id']).to_dict(orient = 'records')
    X_val = dv.transform(X_dict)
    X_val[:, indices] = scaler.transform(X_val[:, indices])
    y_pred = model.predict_proba(X_val)[:, 1]
    return y_pred




In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

best_c = []
for с in [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 100.0]:
    aucs = []
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    for train_inx, val_inx in kfold.split(df_full_train):
        df_train = df_full_train.iloc[train_inx]
        df_val = df_full_train.iloc[val_inx]

        y_train = df_train.loan_status.values
        y_val = df_val.loan_status.values

        del df_train['loan_status']
        del df_val['loan_status']
        dv, indices, scaler, model = train(df_train, y_train, с = с)
        y_pred = prediction(df_val, dv, indices, scaler, model)
        aucs.append(roc_auc_score(y_val, y_pred))
    best_c.append([с, np.mean(aucs)])
print(max(best_c, key=lambda x: x[1]))

/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: Run

[0.1, np.float64(0.9018773896195655)]


/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [78]:
dv, indices, scaler, model = train(df_full_train, y_full, C = 0.1)
y_pred = prediction(df_test, dv, indices, scaler, model)
submission = pd.DataFrame({
    'id': df_test['id'],
    'loan_status': y_pred
})
submission.to_csv('submission.csv', index=False)
submission.head()

/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,id,loan_status
0,58645,0.928413
1,58646,0.009947
2,58647,0.569706
3,58648,0.091341
4,58649,0.380899
